In [2]:
"""
Meridian Data Inspection and Cleaning Pipeline
----------------------------------------------
Place this script in the same folder as:
  Feed_2026_01.zip
  Feed_2026_02.zip
  Feed_2026_03.zip

Requirements:
    pip install pandas

Run:
    python meridian_pipeline.py

The script extracts monthly ZIP files, inspects CSVs, cleans each CSV separately,
and writes results to ./cleaned_data/. Original files are not modified.
"""

from pathlib import Path
import zipfile
import pandas as pd

# Folder containing this script and the monthly ZIP file
BASE_FOLDER = Path.cwd()
OUTPUT_FOLDER = BASE_FOLDER / "cleaned_data"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"Working folder: {BASE_FOLDER}")

# 1. Extract monthly feed ZIP files (if present)
zip_files = sorted(
    p for p in BASE_FOLDER.glob("Feed_20*.zip")
    if p.is_file()
)

for zip_path in zip_files:
    extract_folder = BASE_FOLDER / zip_path.stem
    extract_folder.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_folder)
        print(f"Extracted: {zip_path.name} -> {extract_folder.name}/")
    except zipfile.BadZipFile:
        print(f"WARNING: {zip_path.name} is not a valid ZIP file; skipped.")

# 2. Find input CSV files; ignore this script's output folder
csv_files = sorted(
    p for p in BASE_FOLDER.rglob("*.csv")
    if OUTPUT_FOLDER not in p.parents
    and ".ipynb_checkpoints" not in p.parts
)

if not csv_files:
    raise FileNotFoundError(
        "No CSV files found. Put the Feed_2026_01.zip, Feed_2026_02.zip, "
        "and Feed_2026_03.zip files next to this script, or extract them "
        "into subfolders under this directory."
    )

print(f"\nTotal CSV files found: {len(csv_files)}")

# Column name variations observed in the notebook
COLUMN_MAPPING = {
    "mbr_id": "member_id",
    "svc_from_dt": "service_from_date",
    "svc_to_dt": "service_to_date",
    "allowed_amt": "allowed_amount",
    "paid_amt": "paid_amount",
}

missing_reports = []
cleaning_summary = []
used_output_names = set()

# 3. Process every CSV independently
for file_path in csv_files:
    # Keep IDs and codes as text so leading zeroes are preserved.
    try:
        df = pd.read_csv(file_path, dtype="string")
    except Exception as exc:
        print(f"\nWARNING: Could not read {file_path}: {exc}")
        continue

    original_rows = len(df)
    original_columns = len(df.columns)
    original_duplicates = int(df.duplicated().sum())

    print("\n" + "=" * 70)
    print(f"FILE: {file_path.relative_to(BASE_FOLDER)}")
    print(f"Original shape: {df.shape}")
    print(f"Original duplicate rows: {original_duplicates}")
    print("Original missing values by column:")
    print(df.isna().sum()[df.isna().sum() > 0].to_string() or "None")

    # 4. Standardize column names
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
    )

    # Report duplicate headers rather than silently guessing which to keep
    duplicate_headers = df.columns[df.columns.duplicated()].tolist()
    if duplicate_headers:
        print(
            "WARNING: Duplicate column names after standardization:",
            duplicate_headers,
            "- inspect this file manually."
        )

    # 5. Clean text cells and blank strings
    text_columns = df.select_dtypes(include=["object", "string"]).columns
    for col in text_columns:
        df[col] = df[col].str.strip()
        df[col] = df[col].replace("", pd.NA)

    # 6. Standardize known column name variations
    rename_map = {
        old: new for old, new in COLUMN_MAPPING.items()
        if old in df.columns and new not in df.columns
    }
    df.rename(columns=rename_map, inplace=True)

    # 7. Standardize recognizable date columns
    for col in df.columns:
        if "date" in col.lower() or col.lower().endswith("_dt"):
            original_values = df[col].copy()
            parsed_dates = pd.to_datetime(
                original_values,
                format="mixed",
                errors="coerce"
            )
            formatted = parsed_dates.dt.strftime("%Y-%m-%d")
            # Preserve missing/unparseable values as missing; do not invent dates.
            df[col] = formatted.where(parsed_dates.notna(), pd.NA)

    # 8. Standardize numeric-looking procedure codes to 4 digits
    for col in df.columns:
        if "procedure" in col.lower() and "code" in col.lower():
            df[col] = df[col].map(
                lambda value: value.zfill(4)
                if pd.notna(value) and str(value).isdigit()
                else value
            )

    # 9. Remove only exact duplicate rows
    rows_before_dedup = len(df)
    df = df.drop_duplicates().copy()
    duplicates_removed = rows_before_dedup - len(df)

    # 10. Create unique output name to avoid overwriting same-named monthly files
    relative_parent = file_path.parent.relative_to(BASE_FOLDER)
    prefix_parts = [part for part in relative_parent.parts if part]
    prefix = "_".join(prefix_parts)
    base_name = f"{prefix}_{file_path.stem}" if prefix else file_path.stem
    output_name = f"{base_name}_cleaned.csv"

    counter = 2
    while output_name.lower() in used_output_names:
        output_name = f"{base_name}_{counter}_cleaned.csv"
        counter += 1
    used_output_names.add(output_name.lower())

    output_path = OUTPUT_FOLDER / output_name
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    # 11. Missing-value report after cleaning
    for col in df.columns:
        missing_count = int(df[col].isna().sum())
        missing_reports.append({
            "file_name": str(file_path.relative_to(BASE_FOLDER)),
            "column": col,
            "missing_count": missing_count,
            "missing_percentage": round(
                float(df[col].isna().mean() * 100), 2
            ) if len(df) else 0.0,
        })

    # 12. Summary
    cleaning_summary.append({
        "file_name": str(file_path.relative_to(BASE_FOLDER)),
        "original_rows": original_rows,
        "original_columns": original_columns,
        "original_duplicate_rows": original_duplicates,
        "duplicate_rows_removed": int(duplicates_removed),
        "rows_after_cleaning": len(df),
        "columns_after_cleaning": len(df.columns),
        "missing_values_remaining": int(df.isna().sum().sum()),
        "duplicate_rows_remaining": int(df.duplicated().sum()),
        "cleaned_file": str(output_path),
    })

    print(f"Duplicates removed: {duplicates_removed}")
    print(f"Cleaned shape: {df.shape}")
    print(f"Saved: {output_path}")

# 13. Save reports
summary_path = OUTPUT_FOLDER / "cleaning_summary.csv"
missing_path = OUTPUT_FOLDER / "missing_values_report.csv"

pd.DataFrame(cleaning_summary).to_csv(
    summary_path, index=False, encoding="utf-8-sig"
)
pd.DataFrame(missing_reports).to_csv(
    missing_path, index=False, encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("PIPELINE FINISHED")
print(f"Input CSV files processed: {len(cleaning_summary)}")
print(f"Cleaned files and reports: {OUTPUT_FOLDER}")
print(f"Summary report: {summary_path}")
print(f"Missing-value report: {missing_path}")
print("\nNote: Missing values are reported, not automatically filled.")
print("Only exact duplicate rows are removed; records sharing an ID are retained.")


Working folder: C:\Users\MAYA\Desktop\data  analysis\final project\zip_file
Extracted: Feed_2026_01.zip -> Feed_2026_01/
Extracted: Feed_2026_02.zip -> Feed_2026_02/
Extracted: Feed_2026_03.zip -> Feed_2026_03/

Total CSV files found: 9

FILE: Feed_2026_01\Meridian_Claims_Analytics\02_monthly_feeds\2026_01\claim_lines_2026_01.csv
Original shape: (12914, 9)
Original duplicate rows: 0
Original missing values by column:
modifier    11936
Duplicates removed: 0
Cleaned shape: (12914, 9)
Saved: C:\Users\MAYA\Desktop\data  analysis\final project\zip_file\cleaned_data\Feed_2026_01_Meridian_Claims_Analytics_02_monthly_feeds_2026_01_claim_lines_2026_01_cleaned.csv

FILE: Feed_2026_01\Meridian_Claims_Analytics\02_monthly_feeds\2026_01\claims_header_2026_01.csv
Original shape: (6404, 19)
Original duplicate rows: 0
Original missing values by column:
processed_date         1061
secondary_diagnosis    5609
denial_code            6021
original_claim_id      6362
Duplicates removed: 0
Cleaned shape: (6